# 0. Optional Steps

The steps in this section are optional and used for debug logging

## 0.1 Set the environment to use a proxy

In [ ]:
import os

# Setup proxy through local proxy; bypass proxy for metadata endpoint to avoid any issues with Microsoft authentication libraries
os.environ["HTTP_PROXY"] = "http://127.0.0.1:8000"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:8000"
os.environ["NO_PROXY"] = "169.254.169.254"

# Add private CA root cert to trusted cert bundle for SDKs that use requests and httpx
os.environ["REQUESTS_CA_BUNDLE"] = "/Users/matthewfelton/Library/Preferences/httptoolkit/ca.pem"
os.environ["SSL_CERT_FILE"] = "/Users/matthewfelton/Library/Preferences/httptoolkit/ca.pem"

import os

# Setup proxy through local proxy; bypass proxy for metadata endpoint to avoid any issues with Microsoft authentication libraries
os.environ["HTTP_PROXY"] = "http://127.0.0.1:8000"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:8000"
os.environ["NO_PROXY"] = "169.254.169.254"

# Add private CA root cert to trusted cert bundle for SDKs that use requests and httpx
os.environ["REQUESTS_CA_BUNDLE"] = "/Users/matthewfelton/Library/Preferences/httptoolkit/ca.pem"
os.environ["SSL_CERT_FILE"] = "/Users/matthewfelton/Library/Preferences/httptoolkit/ca.pem"

# 1. Setup the notebook

The steps in this section setup the notebook

## 1.1 Install required libraries

In [ ]:
%pip install -r requirements.txt

## 1.2 Setup a logger

In [ ]:
# Setup a Python logger
import logging
import sys

def configure_logging(level="ERROR"):
    """This function configures logging for code being run based on the specified level.
    Args:
        level (str): The logging level to use (e.g., "DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL").
    """

    try:
        # Convert the level string to uppercase so it matches what the logging library expects
        logging_level = getattr(logging, level.upper(), None)

        # Setup a logging format
        logging.basicConfig(
            level=logging_level,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            handlers=[logging.StreamHandler(sys.stdout)]
        )
    except Exception as e:
        print(f"Failed to set up logging: {e}", file=sys.stderr)
        sys.exit(1) 

configure_logging("ERROR")

## 1.3 Load environment variables

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(".env")

FOUNDRY_MODEL_OPENAI_ENDPOINT = os.getenv("FOUNDRY_MODEL_OPENAI_ENDPOINT")
FOUNDRY_MODEL_APIM_ENDPOINT = os.getenv("FOUNDRY_MODEL_APIM_ENDPOINT")
FOUNDRY_MODEL_ACCOUNT_ENDPOINT = os.getenv("FOUNDRY_MODEL_ACCOUNT_ENDPOINT")
FOUNDRY_MODEL_PROJECT_ENDPOINT = os.getenv("FOUNDRY_MODEL_PROJECT_ENDPOINT")
ENTRA_TENANT_ID = os.getenv("ENTRA_TENANT_ID")
BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID = os.getenv("BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID")
BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET = os.getenv("BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET")
BU2_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID = os.getenv("BU2_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID")
BU2_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET = os.getenv("BU2_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET")
LLM_DEPLOYMENT_NAME = os.getenv("LLM_DEPLOYMENT_NAME")


# 2. Perform a single chat completion

In [ ]:
# Get an Entra token

from azure.identity import ClientSecretCredential, get_bearer_token_provider

credential = ClientSecretCredential(
    client_id=BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID,
    client_secret=BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET,
    tenant_id=ENTRA_TENANT_ID
)

token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default"
)
print(token_provider())


## 2.1 Send to Foundry openai.azure.com endpoint

In [ ]:
from openai import OpenAI

# Create the Azure OpenAI Service client
# 
client = OpenAI(
    base_url=FOUNDRY_MODEL_OPENAI_ENDPOINT,
    api_key=token_provider
)

first_question = "What is the capital of Sweden?"

first_response = client.chat.completions.create(
    model=LLM_DEPLOYMENT_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": first_question}
    ]
)

print(f"User: {first_question}")
print(f"Assistant: {first_response.choices[0].message.content}")


## 2.2 Send to Foundry account-level endpoint (services.ai.azure.com)

In [ ]:
from openai import OpenAI

# Create the Azure OpenAI Service client
# 
client = OpenAI(
    base_url=FOUNDRY_MODEL_ACCOUNT_ENDPOINT,
    api_key=token_provider
)

first_question = "What is the capital of Sweden?"

first_response = client.chat.completions.create(
    model=LLM_DEPLOYMENT_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": first_question}
    ]
)

print(f"User: {first_question}")
print(f"Assistant: {first_response.choices[0].message.content}")


## 2.3 Send to Foundry project-level endpoint (services.ai.azure.com/api/projects)

In [ ]:
from openai import OpenAI

# Create the Azure OpenAI Service client
# 
client = OpenAI(
    base_url=FOUNDRY_MODEL_PROJECT_ENDPOINT,
    api_key=token_provider
)

first_question = "What is the capital of Sweden?"

first_response = client.chat.completions.create(
    model=LLM_DEPLOYMENT_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": first_question}
    ]
)

print(f"User: {first_question}")
print(f"Assistant: {first_response.choices[0].message.content}")


## 2.4 Send to Foundry AI Gateway endpoint

In [ ]:
from openai import OpenAI

# Create the Azure OpenAI Service client
# 
client = OpenAI(
    base_url=FOUNDRY_MODEL_APIM_ENDPOINT,
    api_key=token_provider
)

first_question = "What is the capital of Sweden?"

first_response = client.chat.completions.create(
    model=LLM_DEPLOYMENT_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": first_question}
    ]
)

print(f"User: {first_question}")
print(f"Assistant: {first_response.choices[0].message.content}")


# 3. Perform a multi-step ChatCompletion while preserving the conversation locally

In [ ]:
# Get an Entra token

from azure.identity import ClientSecretCredential, get_bearer_token_provider

credential = ClientSecretCredential(
    client_id=BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_ID,
    client_secret=BU1_ENTRA_SERVICE_PRINCIPAL_CLIENT_SECRET,
    tenant_id=ENTRA_TENANT_ID
)

token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default"
)
print(token_provider())


## 3.1 Send to Foundry openai.azure.com endpoint

In [ ]:
from openai import OpenAI

# Create the Azure OpenAI Service client
# 
client = OpenAI(
    base_url=FOUNDRY_MODEL_OPENAI_ENDPOINT,
    api_key=token_provider
)

# The conversation is kept locally and the full history is sent on every call
messages = [{"role": "system", "content": "You are a helpful assistant."}]

first_question = "What is the capital of Sweden?"
messages.append({"role": "user", "content": first_question})

first_response = client.chat.completions.create(
    model=LLM_DEPLOYMENT_NAME,
    messages=messages
)
messages.append({"role": "assistant", "content": first_response.choices[0].message.content})

# The follow up relies on the earlier turns being in the messages list
follow_up_question = "What is the population?"
messages.append({"role": "user", "content": follow_up_question})

follow_up_response = client.chat.completions.create(
    model=LLM_DEPLOYMENT_NAME,
    messages=messages
)
messages.append({"role": "assistant", "content": follow_up_response.choices[0].message.content})

print(f"User: {first_question}")
print(f"Assistant: {first_response.choices[0].message.content}")
print(f"User: {follow_up_question}")
print(f"Assistant: {follow_up_response.choices[0].message.content}")


## 3.2 Send to Foundry account-level endpoint (services.ai.azure.com)

In [ ]:
from openai import OpenAI

# Create the Azure OpenAI Service client
# 
client = OpenAI(
    base_url=FOUNDRY_MODEL_ACCOUNT_ENDPOINT,
    api_key=token_provider
)

# The conversation is kept locally and the full history is sent on every call
messages = [{"role": "system", "content": "You are a helpful assistant."}]

first_question = "What is the capital of Sweden?"
messages.append({"role": "user", "content": first_question})

first_response = client.chat.completions.create(
    model=LLM_DEPLOYMENT_NAME,
    messages=messages
)
messages.append({"role": "assistant", "content": first_response.choices[0].message.content})

# The follow up relies on the earlier turns being in the messages list
follow_up_question = "What is the population?"
messages.append({"role": "user", "content": follow_up_question})

follow_up_response = client.chat.completions.create(
    model=LLM_DEPLOYMENT_NAME,
    messages=messages
)
messages.append({"role": "assistant", "content": follow_up_response.choices[0].message.content})

print(f"User: {first_question}")
print(f"Assistant: {first_response.choices[0].message.content}")
print(f"User: {follow_up_question}")
print(f"Assistant: {follow_up_response.choices[0].message.content}")


## 3.3 Send to Foundry project-level endpoint (services.ai.azure.com/api/projects)

In [ ]:
from openai import OpenAI

# Create the Azure OpenAI Service client
# 
client = OpenAI(
    base_url=FOUNDRY_MODEL_PROJECT_ENDPOINT,
    api_key=token_provider
)

# The conversation is kept locally and the full history is sent on every call
messages = [{"role": "system", "content": "You are a helpful assistant."}]

first_question = "What is the capital of Sweden?"
messages.append({"role": "user", "content": first_question})

first_response = client.chat.completions.create(
    model=LLM_DEPLOYMENT_NAME,
    messages=messages
)
messages.append({"role": "assistant", "content": first_response.choices[0].message.content})

# The follow up relies on the earlier turns being in the messages list
follow_up_question = "What is the population?"
messages.append({"role": "user", "content": follow_up_question})

follow_up_response = client.chat.completions.create(
    model=LLM_DEPLOYMENT_NAME,
    messages=messages
)
messages.append({"role": "assistant", "content": follow_up_response.choices[0].message.content})

print(f"User: {first_question}")
print(f"Assistant: {first_response.choices[0].message.content}")
print(f"User: {follow_up_question}")
print(f"Assistant: {follow_up_response.choices[0].message.content}")


## 3.4 Send to Foundry AI Gateway endpoint

In [ ]:
from openai import OpenAI, DefaultHttpxClient
from http.cookies import SimpleCookie

received_cookie_values = set()
cookie_checks = []

def report_cookie_exchange(response):
    sent_cookies = SimpleCookie()
    sent_cookies.load(response.request.headers.get("cookie", ""))
    reused_names = sorted(
        name for name, cookie in sent_cookies.items()
        if (name, cookie.value) in received_cookie_values
    )
    received_names = set()
    for header in response.headers.get_list("set-cookie"):
        response_cookies = SimpleCookie()
        response_cookies.load(header)
        for name, cookie in response_cookies.items():
            received_names.add(name)
            received_cookie_values.add((name, cookie.value))
    cookie_checks.append({
        "received": bool(response.headers.get_list("set-cookie")),
        "reused": bool(reused_names)
    })
    print(f"\nCookie check: {response.request.method} {response.request.url.path} ({response.status_code})")
    print(f"  Sent cookie names: {sorted(sent_cookies) or 'none'}")
    print(f"  Received Set-Cookie names: {sorted(received_names) or 'none'}")
    print(f"  Previously received cookies sent back with matching values: {reused_names or 'none'}")

# Create the Azure OpenAI Service client
# 
client = OpenAI(
    base_url=FOUNDRY_MODEL_APIM_ENDPOINT,
    api_key=token_provider,
    http_client=DefaultHttpxClient(event_hooks={"response": [report_cookie_exchange]})
)

# The conversation is kept locally and the full history is sent on every call
messages = [{"role": "system", "content": "You are a helpful assistant."}]

first_question = "What is the capital of Sweden?"
messages.append({"role": "user", "content": first_question})

first_response = client.chat.completions.create(
    model=LLM_DEPLOYMENT_NAME,
    messages=messages
)
messages.append({"role": "assistant", "content": first_response.choices[0].message.content})

# The follow up relies on the earlier turns being in the messages list
follow_up_question = "What is the population?"
messages.append({"role": "user", "content": follow_up_question})

follow_up_response = client.chat.completions.create(
    model=LLM_DEPLOYMENT_NAME,
    messages=messages
)
messages.append({"role": "assistant", "content": follow_up_response.choices[0].message.content})

print(f"User: {first_question}")
print(f"Assistant: {first_response.choices[0].message.content}")
print(f"User: {follow_up_question}")
print(f"Assistant: {follow_up_response.choices[0].message.content}")
print(f"--------------------------------------------")
print("Cookie received:", any(check["received"] for check in cookie_checks))
print("Cookie sent back with matching value:", any(check["reused"] for check in cookie_checks))
